# Privacy Filter BR — Fine-tuning

Fine-tune OpenAI Privacy Filter (1.5B / 50M active) on Brazilian PII dataset.

**Output:** `privacy-filter-br/` — LoRA adapter + new 53-class head.

**Runtime:** A100 (~1-1.5h) | L4 (~2-3h) | T4 (will OOM, not recommended)

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade git+https://github.com/huggingface/transformers.git
!pip install -q --upgrade peft accelerate datasets seqeval torchao
# After install, restart runtime: Runtime → Restart runtime

## 2. Upload dataset

Upload `dataset_br.jsonl` and `dataset_br_holdout.jsonl` to `/content/` via the Files panel,
or run the cell below to upload via the form.

In [ ]:
from google.colab import files
uploaded = files.upload()
import os
for fname in uploaded:
    print(f'{fname}: {os.path.getsize(fname)/1024/1024:.1f} MB')

## 3. Verify GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
    print(f'bfloat16 supported: {torch.cuda.is_bf16_supported()}')

## 4. Label taxonomy

In [ ]:
CATEGORIES = [
    'PRIVATE_PERSON', 'PRIVATE_CPF', 'PRIVATE_CNPJ', 'PRIVATE_RG',
    'PRIVATE_CNH', 'PRIVATE_PIS', 'PRIVATE_TITULO_ELEITOR',
    'PRIVATE_CERTIDAO', 'PRIVATE_IE', 'PRIVATE_EMAIL',
    'PRIVATE_PHONE', 'PRIVATE_ADDRESS', 'PRIVATE_DATE',
]
LABELS = ['O'] + [f'{tag}-{cat}' for cat in CATEGORIES for tag in ['B','I','E','S']]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}
print(f'{len(LABELS)} labels (expected 53)')

## 5. Tokenization & BIOES alignment

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer

BASE_MODEL = 'openai/privacy-filter'
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

def char_spans_to_token_tags(text, entities):
    encoding = tokenizer(text, max_length=MAX_LENGTH, truncation=True,
                          return_offsets_mapping=True, padding=False)
    offsets = encoding['offset_mapping']
    tags = [LABEL2ID['O']] * len(offsets)
    for ent in sorted(entities, key=lambda e: e['start']):
        token_idxs = [
            i for i, (s, e) in enumerate(offsets)
            if s < ent['end'] and e > ent['start'] and s != e
        ]
        if not token_idxs:
            continue
        if len(token_idxs) == 1:
            tags[token_idxs[0]] = LABEL2ID[f"S-{ent['label']}"]
        else:
            tags[token_idxs[0]] = LABEL2ID[f"B-{ent['label']}"]
            tags[token_idxs[-1]] = LABEL2ID[f"E-{ent['label']}"]
            for i in token_idxs[1:-1]:
                tags[i] = LABEL2ID[f"I-{ent['label']}"]
    encoding.pop('offset_mapping')
    encoding['labels'] = tags
    return encoding

def load_jsonl(path):
    examples = []
    with open(path) as f:
        for line in f:
            ex = json.loads(line)
            examples.append(char_spans_to_token_tags(ex['text'], ex['entities']))
    return Dataset.from_list(examples)

train_ds = load_jsonl('/content/dataset_br.jsonl')
eval_ds = load_jsonl('/content/dataset_br_holdout.jsonl')
print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')

## 6. Load model with new 53-class head + LoRA

In [ ]:
from transformers import AutoModelForTokenClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
    trust_remote_code=True,
)

lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['q_proj', 'v_proj'],
    modules_to_save=['score'],  # Privacy Filter calls the head 'score', not 'classifier'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Training

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_labels = [[ID2LABEL[l] for l in lbl if l != -100] for lbl in labels]
    true_preds = [[ID2LABEL[p] for (p, l) in zip(pred, lbl) if l != -100]
                   for pred, lbl in zip(preds, labels)]
    return {
        'precision': precision_score(true_labels, true_preds),
        'recall': recall_score(true_labels, true_preds),
        'f1': f1_score(true_labels, true_preds),
    }

training_args = TrainingArguments(
    output_dir='/content/privacy-filter-br',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

## 8. Final benchmark per category

In [ ]:
preds = trainer.predict(eval_ds)
pred_ids = np.argmax(preds.predictions, axis=2)
true_labels = [[ID2LABEL[l] for l in lbl if l != -100] for lbl in preds.label_ids]
pred_labels = [[ID2LABEL[p] for (p, l) in zip(pred, lbl) if l != -100]
                for pred, lbl in zip(pred_ids, preds.label_ids)]

report = classification_report(true_labels, pred_labels, digits=4)
print(report)

with open('/content/privacy-filter-br/benchmark.txt', 'w') as f:
    f.write(report)

## 9. Save & download

In [ ]:
trainer.save_model('/content/privacy-filter-br')
tokenizer.save_pretrained('/content/privacy-filter-br')

import shutil
shutil.make_archive('/content/privacy-filter-br', 'zip', '/content/privacy-filter-br')

from google.colab import files
files.download('/content/privacy-filter-br.zip')

## 10. Quick test (optional)

In [ ]:
from transformers import pipeline

pipe = pipeline('token-classification', model='/content/privacy-filter-br',
                aggregation_strategy='first')

test = 'Cliente João Silva (CPF 680.075.670-97, RG 27.141.489-3) realizou compra. CNPJ emitente: 72.682.864/0001-41. Contato: maria@empresa.com.br ou (11) 99999-9999.'
for r in pipe(test):
    print(f"{r['entity_group']:25s} {r['word']!r}")